# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim

In [ ]:
from datasets import load_dataset
import pandas as pd
import re       # libreria de expresiones regulares
import string   # libreria de cadena de caracteres
from gensim.models.phrases import Phrases, Phraser
import multiprocessing
from gensim.models import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [37]:
sinopsis = load_dataset("mathigatti/spanish_imdb_synopsis")

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [38]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/usuarios/usuarios.csv")

In [39]:
usuarios

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...
2,U03,Camila,definido,Los padres de él,Mamá a la fuerza,Norbit,Elizabethtown,My Sassy Girl,Una comedia donde la relación entre dos person...
3,U04,Tomás,definido,Matrix,Fahrenheit 451,¡Olvídate de mí!,X-Men,El único,Algo que haga pensar sobre qué es real y qué e...
4,U05,Lucía,definido,La novia cadáver,Spirit: El corcel indomable,Las aventuras de Peabody y Sherman,Los Increíbles,Steamboy,Animación donde el protagonista lucha por su l...
5,U06,Martín,definido,Bienvenidos a Collinwood,El gran golpe,L.A. Confidential,Sympathy for Mr. Vengeance,La otra cara del crimen,Un grupo de personas planea un robo o estafa y...
6,U07,Sofía,definido,Velvet Goldmine,"Cuanto más, ¡mejor!",La vida de bohemia,Cero en conducta,Corazón salvaje,Una película sobre músicos o artistas que vive...
7,U08,Diego,definido,Superdetective en Hollywood,Mission: Impossible,Misión: Imposible 3,"Walker, Texas Ranger",300,Acción directa con un héroe que trabaja solo o...
8,U09,Elena,definido,Viaje a Darjeeling,Mi Idaho privado,Melinda y Melinda,La ciencia del sueño,Un beso,Algo tranquilo sobre personas que intentan rec...
9,U10,Facundo,definido,Sátántangó,Corazón salvaje,Mi Idaho privado,La ciencia del sueño,Sympathy for Mr. Vengeance,"Algo que sea difícil de clasificar, con una ló..."


Visualizamos las queries

In [40]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [41]:
df_pelis = pd.DataFrame(sinopsis['train'])
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

Defino una funcion para limpiar texto

In [42]:
def limpiar_texto(text):
    # pasa las mayusculas del texto a minusculas
    text = text.lower()
    # reemplaza texto entre corchetes por espacio en blanco
    text = re.sub(r'\[.*?¿\]%', ' ', text)
    # reemplaza signos de puntuacion por espacio en blanco
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    # remueve palabras que contienen numeros.
    text = re.sub(r'\w*\d\w*', '', text)
    # remueve caracteres especiales y saltos de linea
    text = re.sub('[‘’“”…«»]', '', text)
    text = re.sub('\n', ' ', text)
    return text

Unificamos las variables relevantes en un texto

In [43]:
df_pelis["texto"] = (
    df_pelis["name"] + " "
    + df_pelis["description"] + " "
    + df_pelis["year"].fillna('').astype(str) + " "
    + df_pelis["director"].fillna('') + " "
    # + df_pelis["genre"] + " " # revisar espanglish
    # + df_pelis["keywords"] # revisar espanglish
)

In [44]:
limpieza = lambda x: limpiar_texto(x)
data_clean = pd.DataFrame(df_pelis["texto"].apply(limpieza))

Agregamos bigramas al corpus (?)

In [63]:
input = [row.split() for row in data_clean["texto"]] # separamos en una lista
phrases = Phrases(input, min_count=10, progress_per=1000)

bigram = Phraser(phrases)

sentences = bigram[input]

## Embedding de peliculas

### Entrenamos modelo World2Vec

> [!!!] falta elegir los parametros del modelo acorde al trabajo.

In [ ]:
cores = multiprocessing.cpu_count()

w2v_model = Word2Vec(min_count=10, # ignora palabras cuya frecuencia es menor a esta
                     window=8, # tamanio de la ventana de contexto
                     vector_size=300, # dimension del embedding
                     sample=6e-5, # umbral para downsamplear palabras muy frecuentes
                     alpha=0.03, # tasa de aprendizaje inicial (entrenamiento de la red neuronal)
                     min_alpha=0.0007, # tasa de aprendizaje minima
                     negative=20, # penalidad de palabras muy frecuentes o poco informaitvas
                     workers=cores) # numero de cores para entrenar el modelo

w2v_model.build_vocab(sentences, progress_per=10000) # construye el vocabulario

### ENTRENA EL MODELO
w2v_model.train(sentences, total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

(1134689, 4444200)

### Calcular vector promedio de cada película

Definimos una funcion que calcula el vector promedio a partir de un texto

In [80]:
def obtener_vector_promedio(texto, modelo):
    palabras = texto.split()

    vectores_palabras = [modelo.wv[palabra] for palabra in palabras if palabra in modelo.wv]

    if not vectores_palabras:
        return np.zeros(modelo.wv.vector_size)

    return np.mean(vectores_palabras, axis=0)

Calculamos el embedding promedio de cada pelicula

In [81]:
embeddings_peliculas = pd.DataFrame(np.array([obtener_vector_promedio(text, w2v_model) for text in data_clean['texto']]))

In [82]:
embeddings_peliculas.head()

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.146794,-0.042294,0.053306,0.066125,-0.013384,-0.001600,0.055941,0.105402,-0.056295,-0.030566,...,0.029926,0.040730,0.091244,0.116058,-0.015166,-0.011909,-0.035844,-0.078724,0.002906,-0.026068
1,0.145235,-0.045676,0.049626,0.062484,-0.011482,0.000597,0.057711,0.100084,-0.057004,-0.027892,...,0.030607,0.040938,0.088686,0.111499,-0.018960,-0.016396,-0.038084,-0.078217,0.003513,-0.024290
2,0.148187,-0.045940,0.044927,0.064177,-0.008945,-0.000907,0.062147,0.104270,-0.060803,-0.024836,...,0.021974,0.039910,0.089306,0.112108,-0.014065,-0.018103,-0.037256,-0.077636,0.002762,-0.023463
3,0.147714,-0.044777,0.048479,0.062730,-0.011249,-0.001652,0.058395,0.102691,-0.057730,-0.026913,...,0.026212,0.040753,0.089100,0.111845,-0.016190,-0.017260,-0.037473,-0.077741,0.004103,-0.024140
4,0.144374,-0.044545,0.049897,0.064219,-0.011292,0.001474,0.057489,0.099644,-0.057793,-0.027617,...,0.031265,0.040851,0.090153,0.112112,-0.018241,-0.018394,-0.039229,-0.078226,0.004913,-0.026323


Estan en orden entonces podemos joinear por index

In [83]:
pelis_embd = embeddings_peliculas.merge(df_pelis[["name","id"]], left_index=True, right_index=True)
pelis_embd.head()

,0,1,2,3,4,5,6,7,8,9,...,992,993,994,995,996,997,998,999,name,id
0,0.146794,-0.042294,0.053306,0.066125,-0.013384,-0.001600,0.055941,0.105402,-0.056295,-0.030566,...,0.091244,0.116058,-0.015166,-0.011909,-0.035844,-0.078724,0.002906,-0.026068,Herida abierta,1
1,0.145235,-0.045676,0.049626,0.062484,-0.011482,0.000597,0.057711,0.100084,-0.057004,-0.027892,...,0.088686,0.111499,-0.018960,-0.016396,-0.038084,-0.078217,0.003513,-0.024290,"Elvira, reina de las tinieblas",2
2,0.148187,-0.045940,0.044927,0.064177,-0.008945,-0.000907,0.062147,0.104270,-0.060803,-0.024836,...,0.089306,0.112108,-0.014065,-0.018103,-0.037256,-0.077636,0.002762,-0.023463,Durmiendo con su enemigo,3
3,0.147714,-0.044777,0.048479,0.062730,-0.011249,-0.001652,0.058395,0.102691,-0.057730,-0.026913,...,0.089100,0.111845,-0.016190,-0.017260,-0.037473,-0.077741,0.004103,-0.024140,Elizabethtown,4
4,0.144374,-0.044545,0.049897,0.064219,-0.011292,0.001474,0.057489,0.099644,-0.057793,-0.027617,...,0.090153,0.112112,-0.018241,-0.018394,-0.039229,-0.078226,0.004913,-0.026323,Godzilla,5


## Embeddings de usuarios

Limpiamos las queries con el mismo proceso de antes

In [84]:
data_clean_users = pd.DataFrame(usuarios["query"].apply(limpieza))

### Embedding de las queries

In [85]:
embeddings_query = pd.DataFrame()

for query in data_clean_users["query"]:
    words = query.split()
    words_embeddings = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    embedding_mean = np.mean(words_embeddings, axis=0)
    embeddings_query = pd.concat([embeddings_query, pd.DataFrame(embedding_mean).T])

embeddings_query.reset_index(drop=True,inplace=True)

In [86]:
embeddings_query = embeddings_query.merge(usuarios[["id"]], left_index=True, right_index=True)
embeddings_query

,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,id
0,0.143971,-0.045180,0.048742,0.064872,-0.008779,0.000640,0.057015,0.102094,-0.058667,-0.027727,...,0.041255,0.085808,0.112409,-0.014701,-0.016107,-0.039461,-0.077014,0.001570,-0.023948,U01
1,0.143244,-0.044017,0.052095,0.064730,-0.011405,-0.000387,0.055211,0.099699,-0.056398,-0.029677,...,0.041003,0.087679,0.116453,-0.017265,-0.017608,-0.040039,-0.079204,0.005680,-0.025421,U02
2,0.141802,-0.044368,0.051420,0.064784,-0.010477,-0.000403,0.053019,0.099107,-0.056375,-0.029916,...,0.042028,0.086413,0.114436,-0.016704,-0.015836,-0.040564,-0.079137,0.003409,-0.024758,U03
3,0.144192,-0.044272,0.045164,0.061206,-0.007873,0.000857,0.059720,0.099863,-0.060835,-0.025406,...,0.040358,0.085561,0.112764,-0.013732,-0.019923,-0.040257,-0.079111,0.003728,-0.024371,U04
4,0.148253,-0.046167,0.047647,0.061314,-0.009991,0.000562,0.061128,0.101742,-0.058893,-0.025772,...,0.039700,0.088029,0.110350,-0.017521,-0.018188,-0.038310,-0.077664,0.004728,-0.023884,U05
5,0.145270,-0.042836,0.053036,0.064729,-0.012688,-0.000102,0.055499,0.100654,-0.058437,-0.030285,...,0.041214,0.089031,0.114590,-0.016659,-0.016746,-0.038837,-0.079644,0.005597,-0.024527,U06
6,0.140280,-0.044626,0.046627,0.061578,-0.008520,-0.000352,0.055876,0.098235,-0.059122,-0.027804,...,0.041828,0.083333,0.113478,-0.013737,-0.019160,-0.040861,-0.078444,0.003911,-0.023400,U07
7,0.145838,-0.044428,0.047915,0.062721,-0.011201,0.000690,0.059598,0.101295,-0.059529,-0.027745,...,0.039475,0.089277,0.111796,-0.016483,-0.018068,-0.038537,-0.077036,0.005373,-0.025169,U08
8,0.143429,-0.043471,0.050952,0.064645,-0.009819,0.000418,0.055290,0.099294,-0.059994,-0.028776,...,0.041828,0.086887,0.116278,-0.014083,-0.018474,-0.041725,-0.078976,0.003595,-0.025332,U09
9,0.143667,-0.043968,0.045136,0.064054,-0.008592,0.000453,0.058120,0.099531,-0.060265,-0.026109,...,0.041305,0.086008,0.114183,-0.010452,-0.019525,-0.040786,-0.077988,0.003466,-0.025989,U10


### Embedding historial

In [87]:
def calcular_embedding_historial(usuario, pelis_embd):
    peliculas_usuario = usuario[['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']].tolist()
    embeddings = []
    for pelicula in peliculas_usuario:
        # check pelicula in pelis_embd
        if pelicula not in pelis_embd['name'].values:
            print(f"Película '{pelicula}' no encontrada en el DataFrame de embeddings.")
            continue # se omite del calculo del embedding del historial
        embedding = pelis_embd[pelis_embd['name'] == pelicula].iloc[0][:-2].to_numpy(dtype=np.float32)
        embeddings.append(embedding)
    historial_embedding = np.mean(embeddings, axis=0)
    return historial_embedding

In [88]:
historiales_embeddings = []
for index, usuario in usuarios.iterrows():
    historial_embedding = calcular_embedding_historial(usuario, pelis_embd)
    historial_embedding = np.append(historial_embedding, usuario['id'])
    
    historiales_embeddings.append(historial_embedding)

historiales_df = pd.DataFrame(historiales_embeddings)

Película 'Rec' no encontrada en el DataFrame de embeddings.
Película 'El secreto de sus ojos' no encontrada en el DataFrame de embeddings.
Película 'El exorcista' no encontrada en el DataFrame de embeddings.
Película 'Intocable' no encontrada en el DataFrame de embeddings.
Película 'Una mente brillante' no encontrada en el DataFrame de embeddings.
Película 'Paddington' no encontrada en el DataFrame de embeddings.


Podemos agregarlas ya que no tiene sentido que falten
> HACER

In [89]:
historiales_df

,0,1,2,3,4,5,6,7,8,9,...,991,992,993,994,995,996,997,998,999,1000
0,0.14623506,-0.04516007,0.048351847,0.06424666,-0.010464055,-0.00047388216,0.05806578,0.10202221,-0.058692504,-0.027334845,...,0.040990315,0.088548794,0.11370988,-0.015133987,-0.017980032,-0.038878806,-0.07777365,0.0036429968,-0.024367545,U01
1,0.14472325,-0.044736512,0.04830913,0.063621424,-0.0112303635,-0.0010571298,0.057289004,0.1016412,-0.057344783,-0.027258897,...,0.04119741,0.08867994,0.11341987,-0.015660888,-0.016848538,-0.03820508,-0.07880967,0.0033888244,-0.024733702,U02
2,0.14575724,-0.045156695,0.048277237,0.06414367,-0.010483318,-0.00045155679,0.057138704,0.1011619,-0.05796177,-0.027429994,...,0.04113535,0.08812006,0.1130458,-0.015593635,-0.017728299,-0.038771972,-0.078385405,0.003379666,-0.024907196,U03
3,0.1458933,-0.04460715,0.048419226,0.06341486,-0.010875084,-0.00028891716,0.058451273,0.1017622,-0.05836023,-0.027359337,...,0.040913437,0.08945651,0.11297293,-0.016930284,-0.016957086,-0.03786295,-0.078818,0.004449089,-0.024632646,U04
4,0.14565693,-0.045288753,0.049766786,0.06397993,-0.01116438,0.00011143558,0.057018578,0.10178812,-0.05725915,-0.028194219,...,0.041157,0.08891979,0.11270831,-0.017010849,-0.017272826,-0.03852147,-0.07881527,0.0038325817,-0.025218165,U05
5,0.14661804,-0.044661336,0.047836654,0.06339827,-0.010893539,-0.0010528748,0.059144627,0.1029382,-0.05850925,-0.026402827,...,0.040356625,0.089736566,0.11377426,-0.01510953,-0.01618309,-0.037781317,-0.07919692,0.0039680456,-0.025299827,U06
6,0.14317836,-0.044840664,0.051341992,0.06389502,-0.012247449,-0.00019416287,0.053919703,0.09905666,-0.056267083,-0.029335152,...,0.04164337,0.0881569,0.11461508,-0.01652832,-0.017209591,-0.03954418,-0.07947104,0.004969136,-0.02590398,U07
7,0.1463215,-0.04432876,0.050430603,0.06440334,-0.012434814,-0.00051028567,0.057650905,0.10256513,-0.057373147,-0.027998894,...,0.040227287,0.09069811,0.113573715,-0.016721252,-0.016528815,-0.037747078,-0.078675136,0.004710041,-0.0257092,U08
8,0.14427002,-0.045098186,0.048120853,0.06295188,-0.010479009,-0.0006224299,0.056274366,0.09986394,-0.05829231,-0.02815414,...,0.042135768,0.08699131,0.11399975,-0.015143663,-0.018238937,-0.039927144,-0.07890929,0.0042758686,-0.024547357,U09
9,0.14474674,-0.044454366,0.047569368,0.06272422,-0.01034542,-0.00021289187,0.05729819,0.10051451,-0.05817182,-0.027467107,...,0.041397043,0.08846551,0.113389775,-0.015276295,-0.017137969,-0.039306156,-0.07932923,0.004070689,-0.025453463,U10


In [ ]:
movie_embeddings = pelis_embd.iloc[:, :-2].to_numpy(dtype=np.float32)

for user_id in embeddings_query['id']:
    query_embedding = embeddings_query[embeddings_query['id'] == user_id].iloc[0][:-1].to_numpy(dtype=np.float32)
    historial_embedding = historiales_df[historiales_df.iloc[:, -1] == user_id].iloc[0][:-1].to_numpy(dtype=np.float32)

    weights = [0.7, 0.3] # peso del embedding de la query y del historial respectivamente
    mean_query_historial_embedding = np.average([query_embedding, historial_embedding], axis=0, weights=weights)
    
    similarities = cosine_similarity(mean_query_historial_embedding.reshape(1, -1), movie_embeddings)
    
    top_10_indices = similarities.argsort()[0][-10:][::-1]
    
    similar_movies_df = pd.DataFrame({
        'movie_name': pelis_embd.loc[top_10_indices, 'name'].values,
        'similarity_score': similarities[0, top_10_indices]
    })
    display(usuarios[usuarios['id'] == user_id])
    print(usuarios[usuarios['id'] == user_id]['query'].values[0])
    print(f"Top 10 Most Similar Movies for User ID {user_id}:")
    display(similar_movies_df)

,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
0,U01,Valentina,definido,Durmiendo con su enemigo,Más allá de la muerte,Desaparecida,¡Olvídate de mí!,The Crazies,Quiero una película donde una mujer enfrenta u...


Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Top 10 Most Similar Movies for User ID U01:


,movie_name,similarity_score
0,Whale rider,0.999946
1,May,0.999934
2,Tesis,0.999932
3,Ojos de fuego,0.999931
4,Rings,0.999929
5,El grito,0.999928
6,Cuando cae la noche,0.999928
7,La chica del puente,0.999928
8,Más fuerte que su destino,0.999927
9,Le père Noël est une ordure,0.999927


,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
1,U02,Rodrigo,definido,Adiós Bafana,Mi pie izquierdo,L.A. Confidential,Érase una vez en América,El juego del halcón,Busco algo basado en hechos reales sobre corru...


Busco algo basado en hechos reales sobre corrupción o poder político
Top 10 Most Similar Movies for User ID U02:


...

Algo que pueda ver con amigos o solo, que no sea muy larga ni muy corta
Top 10 Most Similar Movies for User ID U13:


,movie_name,similarity_score
0,Un vecino con pocas luces,0.999949
1,El rayo verde,0.999931
2,Beautiful Thing,0.999923
3,Zelig,0.999923
4,Virgen a los 40,0.999922
5,Este chico es un demonio,0.999921
6,El pelotón chiflado,0.999921
7,Arlington Road: Temerás a tu vecino,0.999920
8,Tanguy ¿qué hacemos con el niño?,0.999919
9,Te doy mis ojos,0.999919


,id,nombre,tipo_perfil,pelicula_1,pelicula_2,pelicula_3,pelicula_4,pelicula_5,query
13,U14,Paula,ambiguo,Lost in Translation,Transformers,Mamma Mia! La película,Réquiem por un sueño,Paddington,"No tengo ganas de pensar mucho, pero tampoco q..."


No tengo ganas de pensar mucho, pero tampoco quiero algo vacío, algo intermedio
Top 10 Most Similar Movies for User ID U14:


,movie_name,similarity_score
0,Cuando menos te lo esperas... (Something&apos;...,0.999911
1,Lo que cuenta es el final,0.999906
2,Código: KND,0.999905
3,¿Qué pasa con Bob?,0.999901
4,Peep Show,0.999897
5,Persiguiendo a Amy,0.999896
6,L.I.E.,0.999895
7,Los Teleñecos conquistan Manhattan,0.999893
8,Las Vegas,0.999892
9,Antz (Hormigaz),0.999892


## Opcion 1:
 Promedio del query con promedio de pelicula del historial contra promedio de pelicula.

## Opcion 2:
  Ponderar Promedio de query junto con el historial de pelicula, y compararlo con el promedio de pelicula.

## Opcion 3:
  Ponerle un peso al historial de peliculas por orden de visualizacion y compararlo con el promedio de pelicula.